In [1]:
import sys,os
import xml.etree.ElementTree as ET
import xmltodict
import pickle

In [11]:
HMDB_dir_path = r'../spectral_libraries/HMDB/HMDB_Exp_MSMS'
HMDB_MSMS_dir = os.listdir(r'../spectral_libraries/HMDB/HMDB_Exp_MSMS')
HMDB_MSMS = [
    os.path.join(HMDB_dir_path, x)
    for x in HMDB_MSMS_dir
    if not os.path.isdir(os.path.join(HMDB_dir_path, x))
]
HMDB_MET = r'../spectral_libraries/HMDB/hmdb_metabolites.xml'

In [12]:
# hmdb_index = {}
# def handle_metabolite(path, m):
#     acc = m.get('accession')
#     raw_mass = m.get('monisotopic_molecular_weight')
#     if not acc or not raw_mass:
#         return True
#     try:
#         hmdb_index[acc] = {
#             'name': m.get('name'),
#             'formula': m.get('chemical_formula'),
#             'monoisotopic_mass': float(raw_mass),
#             'inchikey': m.get('inchikey'),
#             'smiles': m.get('smiles'),
#         }
#     except ValueError:
#         pass
#     return True

# def get_item(path,m):
    
# with open(HMDB_MET, 'rb') as f:
#     xmltodict.parse(f, item_depth=2, item_callback=handle_metabolite)
# with open('../spectral_libraries/HMDB/hmdb_index.pkl', 'wb') as f:
#     pickle.dump(hmdb_index, f)

hmdb_index = pickle.load(open("../spectral_libraries/HMDB/hmdb_index.pkl",'rb'))

In [18]:
example = HMDB_MSMS[2000]

xmltodict.parse(open(example,'rb'))['ms-ms']['ionization-mode']

'Negative'

In [127]:
import os
import xmltodict
from pathlib import Path
import shutil

def find_adduct_types(msms_files):
    files_with_missing_adducts = []
    adduct_types = []
    for path in msms_files:
        with open(path,'rb') as f:
            data = xmltodict.parse(f)
            
            adduct_type = data['ms-ms'].get("adduct-type",None)
            adduct = data['ms-ms'].get('adduct',None)
            adduct_mass =  data['ms-ms'].get('adduct-mass',None)
            if (adduct_type == {'@nil':'true'}) and (adduct == {'@nil':'true'}) and (adduct_mass == {'@nil':'true'}):
                files_with_missing_adducts.append(path)
    return files_with_missing_adducts
    
    
    
files_with_missing_adducts = find_adduct_types(HMDB_MSMS[:])

dest_dir = Path('../spectral_libraries/HMDB/HMDB_Exp_MSMS/Exp_MSMS_no_adduct_info/')
dest_dir.mkdir(parents=True, exist_ok=True)

for src in files_with_missing_adducts:
    src = Path(src)
    shutil.move(str(src), str(dest_dir / src.name))

def find_missing_msms_links(msms_files, hmdb_index):
    missing = []
    found = 0
    
    for path in msms_files:
        with open(path, 'rb') as f:
            data = xmltodict.parse(f)
        
        db_id = data['ms-ms'].get('database-id',None)
        
        if db_id is None or db_id not in list(hmdb_index.keys()):
            missing.append(db_id)
        else:
            found += 1
    
    return list(set(missing)), found

missing, found = find_missing_msms_links(HMDB_MSMS[:], hmdb_index)
print(f"Matched: {found}, Missing: {len(missing)}")

Matched: 34120, Missing: 0


In [13]:
import re
import xmltodict
from pathlib import Path
from typing import Iterable, Optional

# ---------- Constants ----------
ION_MODE_MAP = {'positive': 'P', 'negative': 'N'}


# Matches: HMDB0014695_ms_ms_spectrum_2249938_experimental.xml
FILENAME_RE = re.compile(r'^HMDB\d+_ms_ms_spectrum_\d+_experimental\.xml$')

# Field write order — Name first, Num Peaks last (peaks follow)
MSP_FIELD_ORDER = [
    'Name',
    'Synonym',
    'DB#',
    'InChIKey',
    'InChI',
    'SMILES',
    'Formula',
    'ExactMass',
    'Precursor_type',
    'Spectrum_type',
    'PrecursorMZ',
    'Instrument_type',
    'Instrument',
    'Ion_mode',
    'Collision_energy',
    'RetentionTime',
    'Splash',
    'DataSource',
    'Comments',
]


In [19]:
def parse_hmdb_msms_record(
    path: str | Path,
    hmdb_index: dict,
    alias_map: Optional[dict] = None,
) -> Optional[dict]:
    """
    Parse one HMDB experimental MS/MS XML into a dict ready for MSP writing.
    
    Parameters
    ----------
    path : path to per-spectrum HMDB XML
    hmdb_index : dict from accession → metabolite metadata
    alias_map : optional dict mapping secondary accessions → primary
    
    Returns
    -------
    dict ready for write_msp_record(), or None if record cannot be built
    """
    path = Path(path)
    fname = path.name

    accession_from_filename, spectrum_id = fname.split("_")[0:5:4]
    
    with open(path, 'rb') as f:
        msms = xmltodict.parse(f, force_list=('ms-ms-peak',))['ms-ms']
    
    # Prefer the database-id from inside the XML over the filename
    accession = msms.get('database-id') or accession_from_filename
    
    # Resolve secondary accessions if we have an alias map
    met = hmdb_index.get(accession)
    if met is None and alias_map is not None:
        primary = alias_map.get(accession)
        if primary:
            met = hmdb_index.get(primary)
    if met is None:
        return None
    
    # Adduct: HMDB stores 'M+H' + '+' separately; MSP wants '[M+H]+'
    adduct = msms.get('adduct')
    adduct_type = msms.get('adduct-type', '') or ''
    precursor_type = f"[{adduct}]{adduct_type}" if adduct else None
    
    # Peaks
    peaks_raw = (msms.get('ms-ms-peaks') or {}).get('ms-ms-peak', []) or []
    peaks = []
    for p in peaks_raw:
        try:
            peaks.append((float(p['mass-charge']), float(p['intensity'])))
        except (KeyError, TypeError, ValueError):
            continue
    
    if not peaks:
        return None  # no point writing an empty spectrum
    
    # Precursor m/z — comes directly from <adduct-mass>
    precursor_mz = msms.get('adduct-mass')
    try:
        precursor_mz = float(precursor_mz) if precursor_mz else None
    except ValueError:
        precursor_mz = None
    
    return {
        'Name': met.get('name'),
        'DB#': f"{accession}_{spectrum_id}",
        'InChIKey': met.get('inchikey'),
        'SMILES': met.get('smiles'),
        'Formula': met.get('formula'),
        'ExactMass': met.get('monoisotopic_mass'),
        'Precursor_type': precursor_type,
        'Spectrum_type': 'MS2',
        'PrecursorMZ': precursor_mz,
        'Ion_mode': ION_MODE_MAP.get((msms.get('ionization-mode') or '').lower()),
        'Collision_energy': msms.get('collision-energy-voltage'),
        'Splash': msms.get('splash-key'),
        'DataSource': 'HMDB_experimental',
        'Comments': f"source_file={fname}",
        'peaks': peaks,
    }
parse_hmdb_msms_record(example,hmdb_index)

{'Name': 'Cladribine',
 'DB#': 'HMDB0014387_2251558',
 'InChIKey': 'PTOAARAWEBMLNO-KVQBGUIXSA-N',
 'SMILES': 'NC1=C2N=CN([C@H]3C[C@H](O)[C@@H](CO)O3)C2=NC(Cl)=N1',
 'Formula': 'C10H12ClN5O3',
 'ExactMass': 285.062866982,
 'Precursor_type': '[M-H]-',
 'Spectrum_type': 'MS2',
 'PrecursorMZ': 284.0556,
 'Ion_mode': 'N',
 'Collision_energy': '60',
 'Splash': 'splash10-001i-1900000000-02d458221e661841d56f',
 'DataSource': 'HMDB_experimental',
 'Comments': 'source_file=HMDB0014387_ms_ms_spectrum_2251558_experimental.xml',
 'peaks': [(66.0098, 20.17),
  (78.0098, 1.13),
  (105.0207, 4.58),
  (107.0363, 2.35),
  (131.0238, 23.08),
  (132.0316, 100.0),
  (158.0471, 3.18),
  (167.0004, 1.65),
  (168.0083, 7.81),
  (194.024, 1.04)]}

In [15]:
# ---------- Writing ----------

def write_msp_record(record: dict, fh) -> None:
    """Write a single MSP record to an open text file handle."""
    for field in MSP_FIELD_ORDER:
        value = record.get(field)
        if value is None or value == '':
            continue
        fh.write(f"{field}: {value}\n")

    peaks = record.get('peaks', [])
    fh.write(f"Num Peaks: {len(peaks)}\n")

    for mz, intensity in peaks:
        fh.write(f"{mz} {intensity}\n")

    fh.write("\n")  # blank line separates records


def write_hmdb_msp(
    msms_files: Iterable[str | Path],
    hmdb_index: dict,
    output_path: str | Path,
    alias_map: Optional[dict] = None,
    verbose: bool = True,
) -> dict:
    """
    Convert HMDB experimental MS/MS XML files into two MSP files split by ion mode.

    Output files are derived from output_path:
        <output_path>_P.msp  — positive mode
        <output_path>_N.msp  — negative mode

    Parameters
    ----------
    msms_files : iterable of paths to HMDB per-spectrum XMLs
    hmdb_index : dict from accession → metabolite metadata
    output_path : base path for output files (without extension)
    alias_map : optional secondary→primary accession map
    verbose : print progress every 1000 records

    Returns
    -------
    dict with counts: {'written_P', 'written_N', 'skipped_no_match',
                       'skipped_no_peaks', 'skipped_bad_filename',
                       'skipped_unknown_mode', 'errors'}
    """
    counts = {
        'written_P': 0,
        'written_N': 0,
        'skipped_no_match': 0,
        'skipped_no_peaks': 0,
        'skipped_bad_filename': 0,
        'skipped_unknown_mode': 0,
        'errors': 0,
    }

    base = Path(output_path)
    out_paths = {'P': Path(f"{base}_P.msp"), 'N': Path(f"{base}_N.msp")}

    with open(out_paths['P'], 'w') as fh_p, open(out_paths['N'], 'w') as fh_n:
        handles = {'P': fh_p, 'N': fh_n}

        for i, path in enumerate(msms_files, 1):
            try:
                record = parse_hmdb_msms_record(path, hmdb_index, alias_map)
            except Exception as e:
                counts['errors'] += 1
                if verbose:
                    print(f"Error parsing {path}: {e}")
                continue

            if record is None:
                fname = Path(path).name
                if not FILENAME_RE.match(fname):
                    counts['skipped_bad_filename'] += 1
                else:
                    counts['skipped_no_match'] += 1
                continue

            if not record.get('peaks'):
                counts['skipped_no_peaks'] += 1
                continue

            mode = record.get('Ion_mode')
            fh = handles.get(mode)
            if fh is None:
                counts['skipped_unknown_mode'] += 1
                continue

            write_msp_record(record, fh)
            counts[f'written_{mode}'] += 1

            if verbose and i % 1000 == 0:
                print(f"Processed {i} files | P: {counts['written_P']}  N: {counts['written_N']}")

    if verbose:
        print(f"\nDone.")
        print(f"  Positive → {out_paths['P']}")
        print(f"  Negative → {out_paths['N']}")
        for k, v in counts.items():
            print(f"  {k}: {v}")

    return counts


In [16]:
counts = write_hmdb_msp(
    msms_files=HMDB_MSMS,
    hmdb_index=hmdb_index,
    output_path='hmdb_experimental',  # produces hmdb_experimental_P.msp and hmdb_experimental_N.msp
    alias_map=None,
    verbose=True,
)


Processed 1000 files | P: 733  N: 267
Processed 2000 files | P: 1468  N: 532
Processed 3000 files | P: 2171  N: 829
Processed 4000 files | P: 2918  N: 1082
Processed 5000 files | P: 3659  N: 1341
Processed 6000 files | P: 4409  N: 1591
Processed 7000 files | P: 5151  N: 1849
Processed 8000 files | P: 5871  N: 2129
Processed 9000 files | P: 6588  N: 2412
Processed 10000 files | P: 7323  N: 2677
Processed 11000 files | P: 8041  N: 2959
Processed 12000 files | P: 8754  N: 3246
Processed 13000 files | P: 9469  N: 3531
Processed 14000 files | P: 10186  N: 3814
Processed 15000 files | P: 10915  N: 4085
Processed 16000 files | P: 11638  N: 4362
Processed 17000 files | P: 12349  N: 4651
Processed 18000 files | P: 13089  N: 4911
Processed 19000 files | P: 13811  N: 5189
Processed 20000 files | P: 14546  N: 5454
Processed 21000 files | P: 15286  N: 5714
Processed 22000 files | P: 16025  N: 5975
Processed 23000 files | P: 16752  N: 6248
Processed 24000 files | P: 17472  N: 6528
Processed 25000 fi

34120